# Options Pricer — Quantum Amplitude Estimation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qumulator/qumulator-sdk/blob/main/notebooks/options_pricer.ipynb)
[![Qumulator](https://img.shields.io/badge/powered%20by-Qumulator-7c6fff.svg)](https://qumulator.com)

**What this notebook does:** Prices a European call option using a quantum circuit on
Qumulator's cloud simulator and compares the result to the Black-Scholes formula.

**No quantum physics knowledge required.** Fill in the option parameters below and press *Run All*.

### What is a European call option?
A call option gives you the right to buy a stock at a fixed price (the **strike** K)
on a future date (the **expiry** T). If the stock price S_T at expiry is above K,
you profit S_T - K. If it is below, the option expires worthless.

**The pricing problem:** We need the expected value of max(S_T - K, 0) under the
risk-neutral log-normal distribution of S_T, discounted back to today.

### Why quantum?
Classical Monte Carlo estimates this expectation at O(1/sqrt(M)) error for M samples.
Quantum amplitude estimation (QAE) encodes the entire distribution into a quantum state
and extracts the expected payoff from a single measurement sweep — converging at
O(1/M), a quadratic speedup over classical methods.

This notebook implements the core QAE step: a quantum circuit encodes the log-normal
distribution and a payoff-weighted ancilla qubit whose measurement probability *is*
the (normalised) option price.


In [ ]:
# Set your API key ---------------------------------------------------------------
# Free key (no credit card): https://qumulator.com  or run:
#   curl -s -X POST https://api.qumulator.com/keys \
#        -H 'Content-Type: application/json' \
#        -d '{"name":"my-key"}' | python -m json.tool
import os
API_KEY = os.environ.get("QUMULATOR_API_KEY", "YOUR_KEY_HERE")
API_URL = "https://api.qumulator.com"


In [ ]:
%pip install qumulator-sdk --quiet
from qumulator import QumulatorClient
import numpy as np
import time
from scipy.linalg import qr
from scipy.stats import norm

client = QumulatorClient(api_url=API_URL, api_key=API_KEY)
print("SDK ready.")


## Step 1 — Option parameters

| Parameter | Symbol | Meaning |
|-----------|--------|---------|
| `S` | Spot price | Current stock price (USD) |
| `K` | Strike price | Price you can buy the stock at expiry |
| `T` | Time to expiry | In years (0.25 = 3 months) |
| `r` | Risk-free rate | Annual rate (0.05 = 5%) |
| `sigma` | Volatility | Annualised sigma (0.20 = 20%) |


In [ ]:
# Option parameters — edit these
S     = 100.0   # current stock price ($)
K     = 100.0   # strike price — at-the-money
T     = 0.25    # time to expiry in years  (0.25 = 3 months)
r     = 0.05    # risk-free interest rate  (5% per year)
sigma = 0.20    # annualised volatility  (20%)

# Quantum circuit parameters
N_BINS  = 16    # price bins  (4 register qubits — exact statevector mode)
N_SHOTS = 8192  # measurement shots
SEED    = 42

print(f"Option:  S={S}, K={K}, T={T}y, r={r:.0%}, sigma={sigma:.0%}")
print(f"Circuit: {N_BINS} price bins ({int(np.log2(N_BINS))} register qubits + 1 ancilla = {int(np.log2(N_BINS))+1} qubits total)")


## Step 2 — Classical Black-Scholes reference


In [ ]:
def black_scholes_call(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

bs_price = black_scholes_call(S, K, T, r, sigma)
print(f"Black-Scholes price: ${bs_price:.4f}")


## Step 3 — Build the quantum circuit

**5-qubit circuit** (`statevector` mode — no depth restriction):

- **Qubits 0-3** (register): encode 16 discrete log-normal price bins at expiry
- **Qubit 4** (ancilla): P(|1>) = normalised expected payoff

Construction steps:
1. **State-prep unitary** U on qubits 0-3: U|0000> = sum_i sqrt(p_i) |i>  (log-normal weights)
2. **Payoff unitary** V on all 5 qubits: rotates ancilla by 2*arcsin(sqrt(f_i/f_max)) for each bin i
3. Combined gate U_full = V @ kron(U, I_ancilla) applied as a single 32x32 unitary
4. Measurement: P(ancilla=1) x f_max x exp(-rT) = option price


In [ ]:
# 1. Discretise the log-normal price distribution into N_BINS bins
n_reg  = int(np.log2(N_BINS))
mu_ln  = (r - 0.5 * sigma**2) * T       # log-space mean
sig_ln = sigma * np.sqrt(T)              # log-space std dev

log_S_min = np.log(S) + mu_ln - 4 * sig_ln
log_S_max = np.log(S) + mu_ln + 4 * sig_ln
log_S_mid = np.linspace(log_S_min, log_S_max, N_BINS)

S_bins    = np.exp(log_S_mid)
bin_width = (log_S_max - log_S_min) / N_BINS
pdf_vals  = (1.0 / (sig_ln * np.sqrt(2 * np.pi))) * np.exp(
    -0.5 * ((log_S_mid - (np.log(S) + mu_ln)) / sig_ln) ** 2)
probs = np.clip(pdf_vals * bin_width, 0, None)
probs /= probs.sum()

payoffs = np.maximum(S_bins - K, 0.0)

print(f"Price range: ${S_bins[0]:.1f} - ${S_bins[-1]:.1f}")
print(f"In-the-money bins: {(payoffs > 0).sum()} / {N_BINS}")
mc_estimate = np.dot(probs, payoffs) * np.exp(-r * T)
print(f"Discretised MC estimate: ${mc_estimate:.4f}  (Black-Scholes: ${bs_price:.4f})")


In [ ]:
# 2. State-preparation unitary: U|0> = sum_i sqrt(p_i) |i>
def state_prep_unitary(probs):
    n = len(probs)
    target = np.sqrt(probs).reshape(-1, 1)
    rng = np.random.default_rng(0)
    rand_mat = rng.standard_normal((n, n))
    rand_mat[:, 0] = target.ravel()
    Q_mat, _ = qr(rand_mat)
    # Ensure first column points in the same direction as target
    if np.dot(Q_mat[:, 0], target.ravel()) < 0:
        Q_mat[:, 0] *= -1
    return Q_mat.real

U_prep = state_prep_unitary(probs)
assert np.allclose(U_prep[:, 0], np.sqrt(probs), atol=1e-10), "State-prep column mismatch"
print(f"State-prep unitary {U_prep.shape}")
print(f"  Unitarity error : {np.max(np.abs(U_prep.T @ U_prep - np.eye(N_BINS))):.2e}")


In [ ]:
# 3. Payoff unitary — block-diagonal RY rotations conditioned on price bin
f_max    = payoffs.max()
n_total  = N_BINS * 2        # register x ancilla space = 16 x 2 = 32
V_payoff = np.zeros((n_total, n_total))

for i in range(N_BINS):
    c_i     = payoffs[i] / f_max if f_max > 0 else 0.0
    theta_i = 2.0 * np.arcsin(np.sqrt(np.clip(c_i, 0, 1)))
    cos_t   = np.cos(theta_i / 2)
    sin_t   = np.sin(theta_i / 2)
    V_payoff[2*i,   2*i  ] =  cos_t
    V_payoff[2*i+1, 2*i  ] =  sin_t
    V_payoff[2*i,   2*i+1] = -sin_t
    V_payoff[2*i+1, 2*i+1] =  cos_t

assert np.allclose(V_payoff.T @ V_payoff, np.eye(n_total), atol=1e-10), "Payoff unitary not unitary"

# 4. Full 32x32 combined unitary for the 5-qubit circuit
U_prep_full = np.kron(U_prep, np.eye(2))   # lift state-prep to register+ancilla space
U_full      = V_payoff @ U_prep_full        # apply payoff rotation after state prep

# Analytic check: P(ancilla=1) gives the expected (normalised) payoff
psi                = U_full[:, 0]           # state after applying U_full to |00000>
p_ancilla_1_theory = float(np.sum(np.abs(psi[1::2])**2))
option_price_theory = p_ancilla_1_theory * f_max * np.exp(-r * T)

print(f"Full unitary {U_full.shape}")
print(f"  Unitarity error  : {np.max(np.abs(U_full.conj().T @ U_full - np.eye(n_total))):.2e}")
print(f"Analytic P(ancilla=1) : {p_ancilla_1_theory:.6f}")
print(f"Option price (analytic): ${option_price_theory:.4f}")
print(f"Black-Scholes          : ${bs_price:.4f}")


## Step 4 — Submit to Qumulator


In [ ]:
n_qubits_circuit = n_reg + 1   # 4 register + 1 ancilla = 5 qubits total

eng = client.circuit.engine(n_qubits=n_qubits_circuit, mode="statevector")
eng.apply("unitary", list(range(n_qubits_circuit)), params=[U_full])

print(f"Circuit: {n_qubits_circuit} qubits, 1 unitary gate ({U_full.shape[0]}x{U_full.shape[0]})")
print(f"Submitting with {N_SHOTS} shots (seed={SEED})...")
t0 = time.perf_counter()
result = eng.run(shots=N_SHOTS, seed=SEED, return_probabilities=True)
elapsed = time.perf_counter() - t0
print(f"Completed in {elapsed:.2f}s")


In [ ]:
# Extract option price from measurement counts
counts             = result.counts
total_shots        = sum(counts.values())
ancilla_1_shots    = sum(v for k, v in counts.items() if k[-1] == "1")
p_ancilla_measured = ancilla_1_shots / total_shots
quantum_price      = p_ancilla_measured * f_max * np.exp(-r * T)
error_pct          = abs(quantum_price - bs_price) / bs_price * 100

print()
print("=" * 52)
print("  EUROPEAN CALL OPTION PRICE")
print("=" * 52)
print(f"  S={S}, K={K}, T={T}y, sigma={sigma:.0%}, r={r:.0%}")
print("-" * 52)
print(f"  Black-Scholes (exact)    : ${bs_price:.4f}")
print(f"  Quantum (sampled counts) : ${quantum_price:.4f}  ({error_pct:.2f}% error)")
print("-" * 52)
print(f"  Shots: {total_shots:,}   P(ancilla=1): {p_ancilla_measured:.5f}")
print("=" * 52)


## Step 5 — Visualise


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(15, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.35)
bar_w = (S_bins[-1] - S_bins[0]) / N_BINS * 0.85

# ── Panel 1: log-normal price distribution
ax1 = fig.add_subplot(gs[0, 0])
ax1.bar(S_bins, probs,
        width=bar_w,
        color=["#ff6b6b" if s > K else "#7c6fff" for s in S_bins],
        alpha=0.85, edgecolor="white", linewidth=0.4)
ax1.axvline(K, color="white", linestyle="--", linewidth=1.5, label=f"Strike K=${K:.0f}")
ax1.axvline(S, color="#aaffaa", linestyle=":", linewidth=1.5, label=f"Spot S=${S:.0f}")
ax1.set_title("Log-Normal Price Distribution\n(red = ITM, purple = OTM)", color="white")
ax1.legend(fontsize=9)
ax1.set_facecolor("#1a1a2e"); ax1.tick_params(colors="white")
ax1.set_xlabel("S_T ($)", color="white"); ax1.set_ylabel("Probability", color="white")

# ── Panel 2: call payoff profile
ax2 = fig.add_subplot(gs[0, 1])
ax2.bar(S_bins, payoffs, width=bar_w, color="#ff6b6b", alpha=0.85, edgecolor="white", linewidth=0.4)
ax2.axvline(K, color="white", linestyle="--", linewidth=1.5, label=f"Strike K=${K:.0f}")
ax2.set_title("Call Option Payoff  max(S_T - K, 0)", color="white")
ax2.legend(fontsize=9)
ax2.set_facecolor("#1a1a2e"); ax2.tick_params(colors="white")
ax2.set_xlabel("S_T ($)", color="white"); ax2.set_ylabel("Payoff ($)", color="white")

# ── Panel 3: quantum ancilla response per bin
ax3 = fig.add_subplot(gs[1, 0])
anc_probs = [float(np.abs(psi[2*i+1])**2) for i in range(N_BINS)]
ax3.bar(S_bins, anc_probs, width=bar_w, color="#7c6fff", alpha=0.85, edgecolor="white", linewidth=0.4)
ax3.axvline(K, color="white", linestyle="--", linewidth=1.5)
ax3.set_title("Quantum Ancilla Response\n(area = option price / f_max)", color="white")
ax3.set_facecolor("#1a1a2e"); ax3.tick_params(colors="white")
ax3.set_xlabel("S_T ($)", color="white"); ax3.set_ylabel("P(ancilla=1 | bin i)", color="white")

# ── Panel 4: convergence comparison
ax4 = fig.add_subplot(gs[1, 1])
shot_range = [64, 128, 256, 512, 1024, 2048, 4096, 8192]
rng_mc     = np.random.default_rng(1)
cls_err    = []
for n in shot_range:
    samp = [np.mean(np.maximum(
                S * np.exp(mu_ln + sig_ln * rng_mc.standard_normal(n)) - K, 0.0
            )) * np.exp(-r * T)
            for _ in range(40)]
    cls_err.append(np.std(samp))
q_err  = [np.sqrt(p_ancilla_exact * (1 - p_ancilla_exact) / n) * f_max * np.exp(-r * T)
          for n in shot_range]
n_arr  = np.array(shot_range, float)

ax4.loglog(shot_range, cls_err, "o-", color="#ff6b6b", label="Classical MC  O(1/sqrt(M))", lw=2)
ax4.loglog(shot_range, q_err,   "s-", color="#7c6fff", label="Quantum  O(1/sqrt(M))",       lw=2)
ax4.loglog(shot_range, cls_err[0] * np.sqrt(shot_range[0]) / np.sqrt(n_arr),
           "--", color="#aaaaaa", alpha=0.5, label="O(1/sqrt(M)) guide")
ax4.set_xlabel("Shots M", color="white"); ax4.set_ylabel("Price std error ($)", color="white")
ax4.set_title("Convergence comparison\n(full iterative QAE achieves O(1/M))", color="white")
ax4.legend(fontsize=9)
ax4.set_facecolor("#1a1a2e"); ax4.tick_params(colors="white")

fig.patch.set_facecolor("#0f0f23")
for ax in [ax1, ax2, ax3, ax4]:
    for spine in ax.spines.values():
        spine.set_edgecolor("#444466")

plt.suptitle(
    f"Quantum Options Pricer  |  S={S}  K={K}  T={T}y  sigma={sigma:.0%}\n"
    f"Black-Scholes: ${bs_price:.4f}  |  Quantum (from probs): ${quantum_price_exact:.4f}",
    color="white", fontsize=13)
plt.savefig("options_pricer_output.png", dpi=120, bbox_inches="tight", facecolor="#0f0f23")
plt.show()
print("Plot saved: options_pricer_output.png")


## Step 6 — Sensitivity analysis (Greeks)


In [ ]:
# Compute the option price analytically from the circuit (no API call) for Greeks sweeps
def quantum_option_price_local(S_val, K_val, T_val, r_val, sigma_val, n_bins=16):
    mu   = (r_val - 0.5 * sigma_val**2) * T_val
    sig  = sigma_val * np.sqrt(T_val)
    lmin = np.log(S_val) + mu - 4 * sig
    lmax = np.log(S_val) + mu + 4 * sig
    lmid = np.linspace(lmin, lmax, n_bins)
    Sb   = np.exp(lmid)
    bw   = (lmax - lmin) / n_bins
    pdf  = np.exp(-0.5 * ((lmid - (np.log(S_val) + mu)) / sig) ** 2) / (sig * np.sqrt(2 * np.pi))
    prb  = np.clip(pdf * bw, 0, None); prb /= prb.sum()
    pays = np.maximum(Sb - K_val, 0.0)
    fm   = pays.max()
    if fm == 0:
        return 0.0
    Up   = state_prep_unitary(prb)
    Vp   = np.zeros((n_bins * 2, n_bins * 2))
    for i in range(n_bins):
        th = 2.0 * np.arcsin(np.sqrt(np.clip(pays[i] / fm, 0, 1)))
        c, s = np.cos(th / 2), np.sin(th / 2)
        Vp[2*i:2*i+2, 2*i:2*i+2] = [[c, -s], [s, c]]
    Ua = Vp @ np.kron(Up, np.eye(2))
    ps = Ua[:, 0]
    return float(np.sum(np.abs(ps[1::2])**2)) * fm * np.exp(-r_val * T_val)

print("Computing Delta curve (sensitivity to spot price S)...")
S_range     = np.linspace(70, 130, 13)
q_prices_S  = [quantum_option_price_local(s, K, T, r, sigma) for s in S_range]
bs_prices_S = [black_scholes_call(s, K, T, r, sigma) for s in S_range]

print("Computing Vega curve (sensitivity to volatility sigma)...")
sig_range   = np.linspace(0.05, 0.50, 13)
q_prices_v  = [quantum_option_price_local(S, K, T, r, s) for s in sig_range]
bs_prices_v = [black_scholes_call(S, K, T, r, s) for s in sig_range]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor="#0f0f23")
configs = [
    (axes[0], S_range,       q_prices_S, bs_prices_S, "Spot Price S ($)",      "Delta Curve"),
    (axes[1], sig_range*100, q_prices_v, bs_prices_v, "Volatility sigma (%)",  "Vega Curve"),
]
for ax, x_vals, q_vals, bs_vals, xlabel, title in configs:
    ax.plot(x_vals, bs_vals, "o-",  color="#ff6b6b", label="Black-Scholes", lw=2)
    ax.plot(x_vals, q_vals,  "s--", color="#7c6fff", label="Quantum circuit", lw=2)
    ax.set_xlabel(xlabel, color="white"); ax.set_ylabel("Option Price ($)", color="white")
    ax.set_title(title, color="white")
    ax.legend(fontsize=9); ax.set_facecolor("#1a1a2e"); ax.tick_params(colors="white")
    for spine in ax.spines.values(): spine.set_edgecolor("#444466")

fig.patch.set_facecolor("#0f0f23")
plt.suptitle("Greeks: Quantum Circuit vs Black-Scholes", color="white", fontsize=13)
plt.tight_layout()
plt.savefig("options_greeks_output.png", dpi=120, bbox_inches="tight", facecolor="#0f0f23")
plt.show()
print("Plot saved: options_greeks_output.png")


## Summary

The quantum circuit prices the option by encoding the entire probability distribution
into a superposition of price states. The ancilla qubit accumulates the weighted payoff
across all bins simultaneously — a single circuit execution.

**Try it yourself:**
- Change `K` to 80 (deep ITM) or 120 (deep OTM) and re-run
- Set `N_BINS = 64` (6 qubits) for higher discretisation accuracy
- Change `sigma` to see how the Vega curve shifts

**Powered by [Qumulator](https://qumulator.com)** — quantum simulation on classical hardware.
